# 23 — Σύγκριση θεμάτων στο dev

Οι **500 τελικές ανθρώπινες ετικέτες έχουν επικυρωθεί**. Εκπαιδεύει τη logistic
regression στις 300 train και συγκρίνει στο ίδιο **100-review dev** με το
καρφιτσωμένο Qwen 1.5B. Προαιρετικά προσθέτει Microsoft-Decision-1 μέσω OpenRouter.

**Fresh Colab GPU session.** Χρειάζεται το `decision_topics_inputs_20261012.zip`
που ετοιμάστηκε μετά τον έλεγχο των φύλλων. Περιέχει ανθρώπινες ετικέτες και
κείμενα: μην το δημοσιεύσεις σε GitHub ή Space.

Η τοπική πρώτη logistic regression προέβλεψε `other` και στις 100 dev κριτικές
(accuracy 64%, macro-F1 0.1561). Εδώ επανεκτελείται στο κοινό pinned περιβάλλον.
**Μόνο dev. Δεν τρέχει το τελικό test, δεν επιλέγει μοντέλο και δεν ενημερώνει Space.**
Τα dev-fitted calibration scores είναι in-sample diagnostics.
[Πρωτόκολλο](https://github.com/krimits/hotel-review-nlp/blob/claude/funny-rubin-31xgtb/docs/experiments/decision_topic_benchmark/PROTOCOL.md).

## 1. Ρυθμίσεις και χρέωση

Με `ALLOW_MICROSOFT=False` τρέχουν μόνο logistic + Qwen, χωρίς provider key.
Για τρία σκέλη, θέσε `ALLOW_MICROSOFT=True`, επίλεξε όριο USD και πρόσθεσε
**νέο, capped key** ως Colab Secret `OPENROUTER_API_KEY`, με πρόσβαση στο notebook.
Αυτό στέλνει **έως 100 dev κριτικές** στο OpenRouter/Microsoft· ελέγχεις πρώτα
τους όρους δεδομένων. Δεν στέλνει labels, notes ή hotel metadata.

Το 0.10 USD είναι παράδειγμα ορίου παρατηρούμενου κόστους, όχι έγκριση χρέωσης
ή εγγύηση λογαριασμού: ένα αίτημα μπορεί να το υπερβεί. Χωρίς γνωστό κόστος
σταματά πριν από το επόμενο αίτημα. Δεν γίνονται retries.

In [ ]:
from pathlib import Path
SOURCE_COMMIT = "fe51104b73e0ad8c96ffe903a4ef5c63d1f16f39"
ALLOW_MICROSOFT = False
MAX_COST_USD = 0.10
BASE = Path("/content/hotel-decision-topics-dev-v1")
REPO = BASE / "repo"
ENV = BASE / "env"
PYTHON = ENV / "bin/python"
INPUTS = BASE / "inputs"
OUT = REPO / "runs/decision_topics/colab_dev"
ARMS = ["logistic", "qwen"] + (["decision"] if ALLOW_MICROSOFT else [])
print("Σκέλη:", ARMS)


## 2. Απομονωμένη εγκατάσταση και πραγματικά imports

Εγκαθιστά τις απαιτούμενες εκδόσεις στο child venv, χωρίς τα global Gradio/diffusers.
Ελέγχει GPU, πραγματικές εκδόσεις, module origins και Qwen/Transformers imports
πριν από βάρη ή API κλήσεις.

In [11]:
import json
import math
import os
import shutil
import subprocess
import sys

BASE.mkdir(parents=True, exist_ok=True)
clean_env = {k: v for k, v in os.environ.items()}

try:
    import virtualenv
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)

logs = []
def run(command, cwd=None):
    result = subprocess.run([str(x) for x in command], cwd=cwd, env=clean_env,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    logs.append(result.stdout)
    (BASE / "setup_execution.txt").write_text("\n".join(logs))
    if result.returncode:
        print(result.stdout[-6000:])
        raise RuntimeError(f"Setup failed: exit {result.returncode}; see setup_execution.txt")
    return result.stdout

if not REPO.exists():
    run(["git", "clone", "--no-checkout", "https://github.com/krimits/hotel-review-nlp.git", REPO])
    run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO)
assert run(["git", "rev-parse", "HEAD"], cwd=REPO).strip() == SOURCE_COMMIT

# Αν ο φάκελος ENV υπάρχει αλλά δεν είναι λειτουργικός (λείπει το python ή το pip), τον διαγράφουμε
PIP_PATH = ENV / "bin/pip"
if ENV.exists() and (not PYTHON.exists() or not PIP_PATH.exists()):
    shutil.rmtree(ENV)

if not ENV.exists():
    # Δημιουργία καθαρού virtual environment μέσω virtualenv
    run([sys.executable, "-m", "virtualenv", ENV])
    run([PYTHON, "-m", "pip", "install", "--upgrade", "pip"])

run([PYTHON, "-I", "-m", "pip", "--isolated", "install", "--no-user", "-r",
     REPO / "configs/decision_topic_comparison_requirements.txt"])
preflight = """
import importlib, importlib.metadata, json, pathlib, sys
expected = {"numpy":"2.5.4", "scipy":"1.18.1", "scikit-learn":"1.9.1",
            "torch":"2.11.0", "transformers":"4.56.2", "huggingface-hub":"0.36.2", "accelerate":"1.10.1"}
assert pathlib.Path(sys.prefix).resolve() == pathlib.Path(sys.argv[1]).resolve()
assert sys.prefix != sys.base_prefix
report = {"executable":sys.executable,"packages":{}}
for name,version in expected.items():
    module=importlib.import_module({"scikit-learn":"sklearn","huggingface-hub":"huggingface_hub"}.get(name,name))
    installed=importlib.metadata.version(name)
    assert installed.split("+")[0] == version == module.__version__.split("+")[0]
    assert pathlib.Path(module.__file__).resolve().is_relative_to(pathlib.Path(sys.prefix).resolve())
    report["packages"][name]={"version":installed,"origin":module.__file__}
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
assert torch.cuda.is_available(), "Use a Colab GPU runtime"
report["gpu"]=torch.cuda.get_device_name(0)
report["native_bf16"]=torch.cuda.is_bf16_supported()
print(json.dumps(report,indent=2))
"""
preflight_json=run([PYTHON, "-I", "-c", preflight, ENV])
(BASE / "runtime_preflight.json").write_text(preflight_json)
print(preflight_json)

Traceback (most recent call last):
  File "<string>", line 16, in <module>
    assert torch.cuda.is_available(), "Use a Colab GPU runtime"
           ~~~~~~~~~~~~~~~~~~~~~~~^^
AssertionError: Use a Colab GPU runtime



RuntimeError: Setup failed: exit 1; see setup_execution.txt

## 3. Ανέβασε το επικυρωμένο ZIP

Επίλεξε **decision_topics_inputs_20261012.zip**. Το αρχικό ZIP του notebook 22
και το Excel δεν είναι αυτά τα inputs. Ο έλεγχος συγκρίνει κάθε αρχείο με τα
ήδη κλειδωμένα hashes και επαναλαμβάνει τον validator των 500 ετικετών.

In [ ]:
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1:
    raise ValueError("Upload exactly the validated input ZIP")
package=BASE / "decision_topics_inputs.zip"
package.write_bytes(next(iter(uploaded.values())))
del uploaded
print(run([PYTHON, "-I", REPO / "scripts/unpack_decision_topic_inputs.py",
           "--package", package, "--output", INPUTS], cwd=REPO))


## 4. Προαιρετικό provider key και εκτέλεση dev

Το Secret διαβάζεται μόνο όταν ενεργοποιήσεις Microsoft. Δεν τυπώνεται,
δεν γράφεται σε κελί, command argument, log ή ZIP. Η επανάληψη δεν αντικαθιστά
μια προηγούμενη μέτρηση. Αποτυχίες διατηρούνται αντί να εξαφανίζονται.

In [ ]:
run_env=dict(clean_env)
if ALLOW_MICROSOFT:
    if not math.isfinite(MAX_COST_USD) or MAX_COST_USD <= 0:
        raise ValueError("Choose a positive USD budget")
    from google.colab import userdata
    provider_key=userdata.get("OPENROUTER_API_KEY")
    if not provider_key:
        raise ValueError("Enable the OPENROUTER_API_KEY Colab Secret")
    run_env["OPENROUTER_API_KEY"]=provider_key
    del provider_key

command=[PYTHON, "-I", REPO / "scripts/benchmark_decision_topics.py", "run",
         "--sample", INPUTS / "sample", "--labels", INPUTS / "labels_rechecked.json",
         "--partition", "dev", "--device", "cuda", "--arms", *ARMS, "--output", OUT]
if ALLOW_MICROSOFT:
    command += ["--allow-external-api", "--max-cost-usd", str(MAX_COST_USD)]
with (BASE / "execution.txt").open("w") as log:
    result=subprocess.run([str(x) for x in command], cwd=REPO, env=run_env,
                          stdout=log, stderr=subprocess.STDOUT)
run_env.pop("OPENROUTER_API_KEY", None)
print("Exit code:", result.returncode)
print((BASE / "execution.txt").read_text()[-2000:])


## 5. Αναφορά και λήψη αποτελεσμάτων

Τα quality scores εμφανίζονται μόνο για ολοκληρωμένα σκέλη. Το calibrated ECE
υπολογίζεται στα ίδια dev δεδομένα όπου προσαρμόστηκε η θερμοκρασία.
Το p95 συγκρίνει διαφορετικές διαδρομές εκτέλεσης, με network time για το API.
Το κόστος τοπικού CPU/GPU παραμένει άγνωστο, χωρίς μηδενική χρέωση να υπονοείται.

In [ ]:
from IPython.display import Markdown, display
report=json.loads((OUT / "run.json").read_text()) if (OUT / "run.json").exists() else None
if report:
    lines=["| Σκέλος | Πλήρες | Raw macro-F1 | Raw ECE | Calibrated ECE (dev fit) | Referrals <0.7 | p95 ms | USD/1k API attempts |",
           "|---|---|---|---|---|---|---|---|"]
    def fmt(value):
        return "unknown / unavailable" if value is None else f"{value:.4f}"
    for arm in ARMS:
        s=report["summary"][arm];raw=s["raw"] or {};cal=s["calibrated"] or {}
        lines.append(f"| {arm} | {s['complete']} | {fmt(raw.get('macro_f1'))} | {fmt(raw.get('ece'))} | "
                     f"{fmt(cal.get('ece'))} | {cal.get('human_review', 'unavailable')} | "
                     f"{fmt(s['p95_request_ms'])} | {fmt(s['cost_usd_per_1000_attempts'])} |")
    display(Markdown("\n".join(lines)))
    print("Execution complete:",report["execution_complete"])
    print("Microsoft requested:",ALLOW_MICROSOFT)
else:
    print("No completed report. Download diagnostics and send them for inspection.")


In [ ]:
import hashlib
import zipfile
output_package=BASE / "decision_topics_dev_comparison.zip"
with zipfile.ZipFile(output_package,"w",zipfile.ZIP_DEFLATED) as archive:
    if OUT.exists():
        for path in sorted(OUT.iterdir()):
            archive.write(path,"comparison/"+path.name)
    for name in ("execution.txt","setup_execution.txt","runtime_preflight.json"):
        if (BASE/name).exists():archive.write(BASE/name,name)
print("ZIP SHA-256:",hashlib.sha256(output_package.read_bytes()).hexdigest())
files.download(str(output_package))


## Επόμενο βήμα

Στείλε το **decision_topics_dev_comparison.zip**, ακόμη και αν υπάρχει αποτυχία.
Το `freeze.json` αφορά μόνο τα σκέλη που πράγματι εκτελέστηκαν. Για την πλήρη
τριπλή σύγκριση χρειάζονται επιτυχημένα logistic, Qwen και Microsoft dev runs.
Η ανεξαρτησία Α/Β δεν επιβεβαιώνεται από τα αρχεία και παραμένει καταγεγραμμένη
ως `unverified`. Μετά τον έλεγχο παγώνουμε το πλήρες configuration για ένα
τελικό test. Η ακρίβεια παραγωγής μέτρων του triage είναι ξεχωριστή αξιολόγηση.